## VISUALIZATION

In [18]:
# General
import numpy as np
from pathlib import Path

# Other
from bokeh.models import Legend
from bokeh.palettes import Category10
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.io import output_notebook
output_notebook()

Loading BokehJS ...

In [19]:
# Directories

outputs = Path(r"C:\Users\Alex\Desktop\Picaso\NN_project\cloudy_spectra_code\outputs")
out_sin = r"C:\Users\Alex\Desktop\Picaso\NN_project\cloudy_spectra_code\outputs\t1500g100f2k2e10.npz"

In [20]:
# Read single file
def _read_npz_perfile(npz_path):
    """Return (lam_um, Fnu, params dict) from a per-case .npz.
       If a master file sneaks in, it uses the first row and warns once."""
    npz_path = Path(npz_path)
    d = np.load(npz_path, allow_pickle=False)
    lam = d["wavelength_um"].astype(float)
    y   = d["y"]
    x   = d["x"]

    if y.ndim == 1 and x.ndim == 1:
        F = y.astype(float)
        Teff, g, fsed, kzz = map(float, x)
        is_master = False
    else:
        # Fallback for master files: take row 0
        F = y[0].astype(float)
        Teff, g, fsed, kzz = map(float, x[0])
        is_master = True
        print(f"[note] '{npz_path.name}' looks like a master file; showing row 0.")

    params = dict(Teff=Teff, g=g, fsed=fsed, kzz=kzz, filename=npz_path.name, is_master=is_master)
    return lam, F, params

# Quick statistics
def inspect_npz_file(npz_path):
    """Print quick stats for a per-case (or first-row master) .npz."""
    lam, F, p = _read_npz_perfile(npz_path)

    print(f"File         : {p['filename']}")
    print(f"Teff [K]     : {p['Teff']:.0f}")
    print(f"g [m/s^2]    : {p['g']:.0f}")
    print(f"f_sed [-]    : {p['fsed']:.2f}")
    print(f"Kzz [cm^2/s] : {p['kzz']:.3e}")
    print(f"λ size       : {lam.size} (micron)")
    print(f"Fν stats     : min={F.min():.6e}, max={F.max():.6e}, median={np.median(F):.6e}")

    return dict(wavelength_um=lam, Fnu=F, **p)

# Plotting
def plot_npz_file(npz_path, logy=False):
    """Plot a single .npz (per-case preferred; master -> row 0)."""
    lam, F, p = _read_npz_perfile(npz_path)

    title = f"{p['filename']} — Teff={p['Teff']:.0f}, g={p['g']:.0f}, f_sed={p['fsed']:.2f}, Kzz={p['kzz']:.2e}"
    src = ColumnDataSource(dict(lam_um=lam, Fnu=F))

    tools = "pan,wheel_zoom,box_zoom,reset,save"
    fig = figure(title=title, x_axis_label="Wavelength (μm)", y_axis_label="Fν (erg cm⁻² s⁻¹ Hz⁻¹)",
                 sizing_mode="stretch_width", height=420, tools=tools, output_backend="canvas")
    if logy:
        fig.y_axis_type = "log"

    fig.add_tools(HoverTool(tooltips=[("λ (μm)", "@lam_um{0.000}"), ("Fν", "@Fnu{0.00e}")], mode="vline"))
    fig.line('lam_um', 'Fnu', source=src, line_width=2)
    show(fig)

In [21]:
inspect_npz_file(out_sin)
plot_npz_file(out_sin, logy=False)

File         : t1500g100f2k2e10.npz
Teff [K]     : 1500
g [m/s^2]    : 100
f_sed [-]    : 2.00
Kzz [cm^2/s] : 2.000e+10
λ size       : 844 (micron)
Fν stats     : min=2.462520e-16, max=1.921761e-06, median=5.525154e-07
